In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the original dataset
df = pd.read_csv("../data/raw/Mathernal_Risk.csv")

# Standardize column-name whitespace
df.columns = df.columns.str.strip()

# Preserve raw observations in memory
raw_df = df.copy(deep=True)

print("Original records:", len(df))
print("Columns:", df.columns.tolist())
print("Risk distribution:")
print(df["Status"].value_counts())

Original records: 6103
Columns: ['Patient ID', 'Name', 'Age', 'Body Temperature(F)', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']
Risk distribution:
Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64


In [2]:
# Clinical plausibility screening

temp_c = (df["Body Temperature(F)"] - 32) * 5 / 9

checks = pd.DataFrame({
    "invalid_age": ~df["Age"].between(10, 60),
    "invalid_sbp": ~df["Systolic Blood Pressure(mm Hg)"].between(60, 250),
    "invalid_dbp": ~df["Diastolic Blood Pressure(mm Hg)"].between(30, 150),
    "invalid_bp_relation": (
        df["Systolic Blood Pressure(mm Hg)"]
        <= df["Diastolic Blood Pressure(mm Hg)"]
    ),
    "invalid_heart_rate": ~df["Heart rate(bpm)"].between(30, 220),
    "invalid_temperature": ~temp_c.between(34, 43)
}, index=df.index)

print("Flagged records by rule:")
print(checks.sum())

print("\nUnique flagged records:", checks.any(axis=1).sum())
print("Potentially retained:", (~checks.any(axis=1)).sum())

Flagged records by rule:
invalid_age             1
invalid_sbp             0
invalid_dbp             1
invalid_bp_relation    19
invalid_heart_rate      0
invalid_temperature     3
dtype: int64

Unique flagged records: 24
Potentially retained: 6079


In [3]:
# STEP 3: Investigate temperature distribution

temp_col = "Body Temperature(F)"

print("Temperature statistics:")
print(df[temp_col].describe())

print("\nTemperature frequencies:")
display(
    df[temp_col]
    .value_counts()
    .sort_index()
    .rename_axis("Temperature (F)")
    .reset_index(name="Records")
)

print("\nTemperatures below 95°F or above 104°F:")
suspect_temp = df[
    (df[temp_col] < 95) |
    (df[temp_col] > 104)
]

print("Flagged:", len(suspect_temp))

display(
    suspect_temp[
        [temp_col, "Status"]
    ].value_counts().reset_index(name="Records")
)

Temperature statistics:
count    6103.000000
mean       98.665574
std         1.590983
min        39.600000
25%        98.600000
50%        98.600000
75%        98.800000
max       104.000000
Name: Body Temperature(F), dtype: float64

Temperature frequencies:


,Temperature (F),Records
0,39.6,1
1,93.0,2
2,93.9,1
3,94.0,10
4,94.1,3
...,...,...
99,103.6,10
100,103.7,8
101,103.8,8
102,103.9,8



Temperatures below 95°F or above 104°F:
Flagged: 43


,Body Temperature(F),Status,Records
0,94.0,high risk,10
1,94.6,high risk,6
2,94.9,high risk,5
3,94.1,high risk,3
4,94.2,high risk,3
5,94.7,high risk,3
6,94.8,high risk,3
7,93.0,high risk,2
8,94.5,high risk,2
9,94.3,high risk,2


In [4]:
# STEP 4: Compare temperature screening rules

temperature_rules = {
    "Below 93.2°F or above 109.4°F (34–43°C)": (
        ~df[temp_col].between(93.2, 109.4)
    ),
    "Below 95°F or above 104°F": (
        ~df[temp_col].between(95, 104)
    ),
    "Below 96°F or above 104°F": (
        ~df[temp_col].between(96, 104)
    ),
    "Below 97°F or above 103°F": (
        ~df[temp_col].between(97, 103)
    )
}

for rule, mask in temperature_rules.items():
    print(f"{rule}: {mask.sum()} records")

Below 93.2°F or above 109.4°F (34–43°C): 3 records
Below 95°F or above 104°F: 43 records
Below 96°F or above 104°F: 282 records
Below 97°F or above 103°F: 655 records


In [5]:
# STEP 5: Candidate reproduction of published exclusions
# Temperature threshold is inferred, not yet confirmed.

paper_checks = pd.DataFrame({
    "age_outlier": ~df["Age"].between(10, 60),

    "temperature_outlier": ~df[
        "Body Temperature(F)"
    ].between(95, 104),

    "diastolic_outlier": ~df[
        "Diastolic Blood Pressure(mm Hg)"
    ].between(30, 150)
}, index=df.index)

print("Flagged by category:")
print(paper_checks.sum())

print("\nUnique flagged records:",
      paper_checks.any(axis=1).sum())

print("\nRecords passing all three checks:",
      (~paper_checks.any(axis=1)).sum())

print("\nRecords flagged by multiple checks:",
      (paper_checks.sum(axis=1) > 1).sum())

Flagged by category:
age_outlier             1
temperature_outlier    43
diastolic_outlier       1
dtype: int64

Unique flagged records: 45

Records passing all three checks: 6058

Records flagged by multiple checks: 0


In [6]:
# STEP 6: Compare candidate paper exclusions
# with our additional BP plausibility checks

paper_flagged = paper_checks.any(axis=1)

aiovy_bp_flagged = checks["invalid_bp_relation"]

print("Candidate paper exclusions:",
      paper_flagged.sum())

print("Additional BP relation flags:",
      aiovy_bp_flagged.sum())

print("BP flags already included in paper exclusions:",
      (paper_flagged & aiovy_bp_flagged).sum())

print("New BP flags beyond paper exclusions:",
      (~paper_flagged & aiovy_bp_flagged).sum())

combined_flagged = paper_flagged | aiovy_bp_flagged

print("\nUnique records flagged by combined rules:",
      combined_flagged.sum())

print("Potentially retained:",
      (~combined_flagged).sum())

Candidate paper exclusions: 45
Additional BP relation flags: 19
BP flags already included in paper exclusions: 0
New BP flags beyond paper exclusions: 19

Unique records flagged by combined rules: 64
Potentially retained: 6039


In [7]:
from pathlib import Path

processed_dir = Path("../data/processed")
reports_dir = Path("../reports")

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Record every exclusion reason
audit = paper_checks.copy()
audit["invalid_bp_relation"] = checks["invalid_bp_relation"]

audit["excluded_paper_candidate"] = paper_checks.any(axis=1)
audit["excluded_aiovy_candidate"] = (
    audit["excluded_paper_candidate"] |
    audit["invalid_bp_relation"]
)

# Use row indices for auditing, not patient names
audit.insert(0, "source_row_index", df.index)

audit.to_csv(
    reports_dir / "cleaning_audit.csv",
    index=False
)

print("Audit saved!")
print(audit.sum(numeric_only=True))

Audit saved!
source_row_index            18620253
age_outlier                        1
temperature_outlier               43
diastolic_outlier                  1
invalid_bp_relation               19
excluded_paper_candidate          45
excluded_aiovy_candidate          64
dtype: int64


In [8]:
# Candidate 1: Reproduces the paper's reported count
paper_cleaned = df.loc[~paper_flagged].copy()

# Candidate 2: Additional AIOVY BP screening
aiovy_cleaned = df.loc[~combined_flagged].copy()

# Remove identifiers from exported analysis data
identifier_cols = ["Patient ID", "Name"]

paper_cleaned = paper_cleaned.drop(
    columns=identifier_cols,
    errors="ignore"
)

aiovy_cleaned = aiovy_cleaned.drop(
    columns=identifier_cols,
    errors="ignore"
)

print("Paper-style cleaned:", paper_cleaned.shape)
print("AIOVY candidate cleaned:", aiovy_cleaned.shape)

assert len(paper_cleaned) == 6058
assert len(aiovy_cleaned) == 6039

Paper-style cleaned: (6058, 9)
AIOVY candidate cleaned: (6039, 9)


In [9]:
def standardize_features(data):
    result = data.copy()

    result = result.rename(columns={
        "Age": "age",
        "Systolic Blood Pressure(mm Hg)": "systolic_bp",
        "Diastolic Blood Pressure(mm Hg)": "diastolic_bp",
        "Heart rate(bpm)": "heart_rate",
        "Blood Glucose(Fasting hour-mg/dl)": "blood_glucose_mmol_l",
        "Status": "risk_level"
    })

    # Convert Fahrenheit to Celsius
    result["body_temperature_c"] = (
        result["Body Temperature(F)"] - 32
    ) * 5 / 9

    # Retain exactly six model inputs and the label
    return result[[
        "age",
        "systolic_bp",
        "diastolic_bp",
        "blood_glucose_mmol_l",
        "body_temperature_c",
        "heart_rate",
        "risk_level"
    ]]

paper_final = standardize_features(paper_cleaned)
aiovy_final = standardize_features(aiovy_cleaned)

print("Paper candidate:", paper_final.shape)
print("AIOVY candidate:", aiovy_final.shape)

display(aiovy_final.head())

Paper candidate: (6058, 7)
AIOVY candidate: (6039, 7)


,age,systolic_bp,diastolic_bp,blood_glucose_mmol_l,body_temperature_c,heart_rate,risk_level
0,20,161,100,5.8,36.388889,91,high risk
1,45,99,94,5.7,36.500000,99,high risk
2,29,129,87,6.4,37.000000,84,mid risk
3,26,161,101,4.5,37.500000,135,high risk
4,38,106,91,4.3,39.166667,51,high risk


In [ ]:
paper_final.to_csv(
    processed_dir / "maternal_paper_candidate_6058.csv",
    index=False
)

aiovy_final.to_csv(
    processed_dir / "maternal_aiovy_candidate_6039.csv",
    index=False
)

print("Saved processed dataset candidates!")

Saved processed dataset candidates!


: 